# 🐍 Module 2 — Data-oriented Python
**ML Engineer Program**

| Section | Content |
|---|---|
| 2.1 | Static typing |
| 2.2 | Dataclasses and Pydantic |
| 2.3 | Data manipulation — Pandas, Numpy |


---
## 2.1 Static typing

### Lesson — basic annotations

In [ ]:
from typing import Optional, Union, List, Dict, Tuple, Set, Any
from typing import Callable, TypeVar, Generic

# ── Simple annotations ───────────────────────────────────────
def add(a: int, b: int) -> int:
    return a + b

def greet(name: str, formal: bool = False) -> str:
    prefix = "Dear" if formal else "Hi"
    return f"{prefix} {name}"

# ── Optional = Union[X, None] ─────────────────────────────────
def find_best_model(models: List[Dict], min_auc: float) -> Optional[Dict]:
    "Returns the best model or None if none exceeds min_auc"
    candidates = [m for m in models if m["auc"] >= min_auc]
    return max(candidates, key=lambda m: m["auc"]) if candidates else None

models = [
    {"name": "lgbm", "auc": 0.94},
    {"name": "rf",   "auc": 0.91},
    {"name": "lr",   "auc": 0.88},
]
print(find_best_model(models, min_auc=0.92))   # lgbm
print(find_best_model(models, min_auc=0.99))   # None

# ── Union ─────────────────────────────────────────────────────
def parse_threshold(value: Union[str, float]) -> float:
    "Accepts '0.5' or 0.5"
    if isinstance(value, str):
        return float(value)
    return value

print(parse_threshold("0.75"))
print(parse_threshold(0.75))

# ── Complex types ───────────────────────────────────────────
FeatureVector = List[float]
FeatureMatrix = List[FeatureVector]
MetricDict    = Dict[str, float]

def evaluate(y_true: List[int], y_pred: List[int]) -> MetricDict:
    tp = sum(t == 1 and p == 1 for t, p in zip(y_true, y_pred))
    fp = sum(t == 0 and p == 1 for t, p in zip(y_true, y_pred))
    fn = sum(t == 1 and p == 0 for t, p in zip(y_true, y_pred))
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return {
        "precision": round(p, 4),
        "recall":    round(r, 4),
        "f1":        round(2*p*r/(p+r), 4) if (p+r) > 0 else 0.0,
    }

print(evaluate([1,1,0,1,0], [1,0,0,1,1]))

# ── Typed tuple ────────────────────────────────────────────────
def split_data(X: FeatureMatrix, ratio: float = 0.8
               ) -> Tuple[FeatureMatrix, FeatureMatrix]:
    n     = int(len(X) * ratio)
    return X[:n], X[n:]

data = [[float(i)] for i in range(10)]
train, test = split_data(data, 0.8)
print(f"train={len(train)} test={len(test)}")

### Lesson — TypeVar, Protocol, Callable

In [ ]:
from typing import TypeVar, Protocol, Callable, Iterator, Generic
from abc import abstractmethod

# ── TypeVar — generics ──────────────────────────────────────
T = TypeVar("T")
N = TypeVar("N", int, float)   # contraint à int ou float

def first(items: list[T]) -> T:
    "Returns the first element, type preserved"
    if not items:
        raise ValueError("Empty list")
    return items[0]

print(first([1, 2, 3]))           # int
print(first(["a", "b", "c"]))     # str
print(first([0.1, 0.2]))          # float

def clamp(value: N, lo: N, hi: N) -> N:
    "Clamps a value between lo and hi"
    return max(lo, min(hi, value))

print(clamp(1.5, 0.0, 1.0))    # 1.0
print(clamp(-3, 0, 10))        # 0

# ── Protocol — typed duck typing ───────────────────────────────
# A Protocol defines an interface without forced inheritance
class Fittable(Protocol):
    def fit(self, X: list, y: list) -> "Fittable": ...

class Predictable(Protocol):
    def predict(self, X: list) -> list: ...

class Scorer(Protocol):
    def score(self, X: list, y: list) -> float: ...

# Any class that implements these methods is compatible
# without having to inherit explicitly

class SimpleClassifier:
    "Implements Fittable + Predictable + Scorer without inheritance"
    def __init__(self, threshold: float = 0.5):
        self.threshold  = threshold
        self._threshold = None

    def fit(self, X: list, y: list) -> "SimpleClassifier":
        self._threshold = sum(y) / len(y)
        return self

    def predict(self, X: list) -> list:
        return [1 if x[0] > self._threshold else 0 for x in X]

    def score(self, X: list, y: list) -> float:
        preds = self.predict(X)
        return sum(p == t for p, t in zip(preds, y)) / len(y)

# ── Typed Callable ─────────────────────────────────────────────
TransformFn = Callable[[float], float]
MetricFn    = Callable[[list, list], float]

def apply_transform(data: list[float], fn: TransformFn) -> list[float]:
    return [fn(x) for x in data]

import math
result = apply_transform([1.0, 4.0, 9.0, 16.0], math.sqrt)
print("sqrt transform:", result)

# Typed transformation pipeline
def make_pipeline(*fns: TransformFn) -> TransformFn:
    import functools
    return functools.reduce(lambda f, g: lambda x: g(f(x)), fns)

pipe = make_pipeline(math.sqrt, lambda x: x * 2, lambda x: round(x, 3))
print("pipeline(16):", pipe(16.0))   # sqrt(16)=4 → 8 → 8.0

### 🏋️ Exercises 2.1

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Fully type these functions
# ════════════════════════════════════════════════════════
# Add correct type annotations to all
# these functions (parameters AND return type).
# Use Optional, Union, List, Dict, Tuple as appropriate.

from typing import Optional, Union, List, Dict, Tuple

# 1a. Returns the max value of a list, or None if empty
def safe_max(values):
    return max(values) if values else None

# 1b. Transforms a list of strings into a list of ints,
#     ignoring non-convertible values
def parse_ints(strings):
    result = []
    for s in strings:
        try:
            result.append(int(s))
        except ValueError:
            pass
    return result

# 1c. Merges two dicts, the second overwrites the first
def merge_configs(base, override):
    return {**base, **override}

# 1d. Returns (mean, std) of a list of floats
def mean_std(values):
    import statistics
    return statistics.mean(values), statistics.stdev(values)

# Tests (do not modify)
print(safe_max([1, 2, 3]))     # 3
print(safe_max([]))            # None
print(parse_ints(["1", "x", "3", "y", "5"]))   # [1, 3, 5]
print(merge_configs({"a": 1}, {"b": 2, "a": 99}))
print(mean_std([1.0, 2.0, 3.0, 4.0, 5.0]))

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def safe_max(values: List[float]) -> Optional[float]:
# def parse_ints(strings: List[str]) -> List[int]:
# def merge_configs(base: Dict[str, Any], override: Dict[str, Any]) -> Dict[str, Any]:
# def mean_std(values: List[float]) -> Tuple[float, float]:

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — TypeVar: generic function batch_apply
# ════════════════════════════════════════════════════════
# Write batch_apply(items, fn, batch_size) typed with TypeVar
# that applies fn on batches of items
# and returns a flattened list of results.
# T = type of input elements
# R = type of output elements

from typing import TypeVar, Callable, List

T = TypeVar("T")
R = TypeVar("R")

def batch_apply(items: List[T],
                fn: Callable[[List[T]], List[R]],
                batch_size: int) -> List[R]:
    pass  # to complete

# Test 1: double ints by batches
result = batch_apply(list(range(10)),
                     fn=lambda batch: [x * 2 for x in batch],
                     batch_size=3)
print("doubled:", result)   # [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]

# Test 2: upper case strings by batches
words = ["hello", "world", "python", "typing", "is", "great"]
result2 = batch_apply(words,
                      fn=lambda batch: [w.upper() for w in batch],
                      batch_size=2)
print("uppercased:", result2)

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def batch_apply(items, fn, batch_size):
#     result = []
#     for i in range(0, len(items), batch_size):
#         result.extend(fn(items[i:i + batch_size]))
#     return result

---
## 2.2 Dataclasses and Pydantic

### Lesson — @dataclass

In [ ]:
from dataclasses import dataclass, field, asdict, astuple, replace
from typing import List, Dict, Optional
import json

# ── Basic dataclass ─────────────────────────────────────────
@dataclass
class Feature:
    name: str
    dtype: str
    missing_rate: float = 0.0
    is_categorical: bool = False

    def __post_init__(self):
        "Validation after __init__"
        valid_dtypes = {"float", "int", "str", "bool"}
        if self.dtype not in valid_dtypes:
            raise ValueError(f"dtype must be one of {valid_dtypes}, got '{self.dtype}'")
        if not 0.0 <= self.missing_rate <= 1.0:
            raise ValueError(f"missing_rate must be in [0,1], got {self.missing_rate}")

f = Feature("age", "int")
print(f)
print("asdict:", asdict(f))

try:
    Feature("x", "unknown_type")
except ValueError as e:
    print(f"Caught: {e}")

# ── field() — complex default values ────────────────────
@dataclass
class ModelConfig:
    name: str
    n_estimators: int       = 100
    learning_rate: float    = 0.05
    features: List[str]     = field(default_factory=list)
    metadata: Dict[str, str]= field(default_factory=dict)
    _fitted: bool           = field(default=False, repr=False, compare=False)

    def __post_init__(self):
        if self.learning_rate <= 0:
            raise ValueError("learning_rate must be > 0")
        # normalize the name
        self.name = self.name.lower().strip()

cfg = ModelConfig("LGBM_v1", features=["age", "tenure"])
print(cfg)
print("JSON:", json.dumps(asdict(cfg), indent=2))

# ── replace() — create a modified copy (immutable pattern) ──
cfg_tuned = replace(cfg, n_estimators=200, learning_rate=0.03)
print("original :", cfg.n_estimators, cfg.learning_rate)
print("tuned    :", cfg_tuned.n_estimators, cfg_tuned.learning_rate)

# ── frozen=True — immutable ────────────────────────────────────
@dataclass(frozen=True)
class ModelVersion:
    name: str
    version: str
    checksum: str

    def __hash__(self):
        return hash((self.name, self.version))

v1 = ModelVersion("lgbm", "1.0.0", "abc123")
v2 = ModelVersion("lgbm", "1.1.0", "def456")
version_registry = {v1: "prod", v2: "staging"}  # hashable car frozen
print("version_registry:", {f"{k.name}@{k.version}": v
                               for k, v in version_registry.items()})

try:
    v1.version = "2.0.0"   # must raise FrozenInstanceError
except Exception as e:
    print(f"Caught: {type(e).__name__}: {e}")

### Lesson — Pydantic

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from pydantic import ValidationError
from typing import Optional, List, Literal
import json

# ── Simple BaseModel ──────────────────────────────────────────
class PredictionRequest(BaseModel):
    age: int = Field(ge=18, le=120, description="Customer age")
    tenure_months: int = Field(ge=0, le=240)
    monthly_charges: float = Field(gt=0.0, le=500.0)
    contract_type: Literal["month-to-month", "one-year", "two-year"]
    nb_products: int = Field(ge=1, le=10, default=1)

# Automatic validation
try:
    req = PredictionRequest(age=35, tenure_months=24,
                            monthly_charges=75.5,
                            contract_type="one-year")
    print("Valid request:", req)
    print("JSON:", req.model_dump_json(indent=2))
except ValidationError as e:
    print("Error:", e)

# Multiple errors caught in one pass
try:
    bad = PredictionRequest(age=200, tenure_months=-5,
                            monthly_charges=0,
                            contract_type="invalid")
except ValidationError as e:
    print(f"{len(e.errors())} validation errors:")
    for err in e.errors():
        print(f"  {err['loc']} → {err['msg']}")

# ── field_validator ───────────────────────────────────────────
class FeatureSchema(BaseModel):
    name: str
    values: List[float]
    dtype: str = "float"

    @field_validator("name")
    @classmethod
    def name_must_be_snake_case(cls, v: str) -> str:
        import re
        if not re.match(r"^[a-z][a-z0-9_]*$", v):
            raise ValueError(f"Feature name must be snake_case, got '{v}'")
        return v

    @field_validator("values")
    @classmethod
    def values_must_be_finite(cls, v: List[float]) -> List[float]:
        import math
        if any(not math.isfinite(x) for x in v):
            raise ValueError("Values must all be finite (no NaN/inf)")
        return v

try:
    f = FeatureSchema(name="monthly_charges", values=[10.0, 20.0, 30.0])
    print(f"Valid feature: {f.name} ({len(f.values)} values)")
except ValidationError as e:
    print(e)

try:
    FeatureSchema(name="Monthly Charges", values=[1.0])  # invalid name
except ValidationError as e:
    print(f"Name error: {e.errors()[0]['msg']}")

# ── model_validator — cross-field validation ─────────
class TrainConfig(BaseModel):
    train_size: float = Field(gt=0.0, lt=1.0)
    val_size: float   = Field(gt=0.0, lt=1.0)
    test_size: float  = Field(gt=0.0, lt=1.0)

    @model_validator(mode="after")
    def sizes_must_sum_to_one(self) -> "TrainConfig":
        total = self.train_size + self.val_size + self.test_size
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"train+val+test must sum to 1.0, got {total:.4f}")
        return self

try:
    cfg = TrainConfig(train_size=0.7, val_size=0.15, test_size=0.15)
    print(f"TrainConfig OK: {cfg}")
except ValidationError as e:
    print(e)

try:
    TrainConfig(train_size=0.8, val_size=0.1, test_size=0.5)  # sum = 1.4
except ValidationError as e:
    print(f"Cross-field error: {e.errors()[0]['msg']}")

# ── Serialization / deserialization ──────────────────────────
class PredictionResponse(BaseModel):
    churn_proba: float = Field(ge=0.0, le=1.0)
    prediction: bool
    model_version: str
    features_used: List[str] = Field(default_factory=list)

resp = PredictionResponse(churn_proba=0.73, prediction=True,
                           model_version="v1.2.0",
                           features_used=["age", "tenure", "charges"])
json_str = resp.model_dump_json()
resp2    = PredictionResponse.model_validate_json(json_str)
print(f"round-trip OK: {resp == resp2}")

# ── When to choose dataclass vs Pydantic ──────────────────────
# dataclass → internal data structures, config, no incoming validation
# Pydantic  → external data validation (API input/output, config files)
#             JSON serialization, YAML/env config parsing
print("dataclass  → config interne, frozen pour hashable, léger")
print("Pydantic   → validation API, parsing JSON, erreurs explicites")

### 🏋️ Exercises 2.2

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Dataclass: feature pipeline
# ════════════════════════════════════════════════════════
# Create a FeaturePipeline dataclass that contains:
# - name : str
# - steps : List[str] (transformation names)
# - input_features : List[str]
# - output_features : List[str] (auto-computed in __post_init__)
#   = input_features + [f"{s}_{f}" for s in steps for f in input_features]
# - n_features_out : int (auto-computed, len(output_features))
# - frozen = False (mutable), but output_features and n_features_out
#   should not be directly settable

from dataclasses import dataclass, field
from typing import List

@dataclass
class FeaturePipeline:
    pass  # to complete

fp = FeaturePipeline(
    name="my_pipeline",
    steps=["lag_1", "rolling_7"],
    input_features=["consumption", "temperature"]
)
print(fp)
print("output_features:", fp.output_features)
print("n_features_out:", fp.n_features_out)
# Expected output_features:
# ['consumption', 'temperature',
#  'lag_1_consumption', 'lag_1_temperature',
#  'rolling_7_consumption', 'rolling_7_temperature']

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# @dataclass
# class FeaturePipeline:
#     name: str
#     steps: List[str]
#     input_features: List[str]
#     output_features: List[str] = field(default_factory=list, init=False, repr=False)
#     n_features_out: int = field(default=0, init=False)
#
#     def __post_init__(self):
#         engineered = [f"{s}_{f}" for s in self.steps for f in self.input_features]
#         self.output_features = self.input_features + engineered
#         self.n_features_out  = len(self.output_features)

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 — Pydantic: ML dataset schema
# ════════════════════════════════════════════════════════
# Create a Pydantic DatasetSchema model that validates:
# - name : str (snake_case only)
# - n_samples : int (>= 100)
# - n_features : int (>= 1)
# - target_col : str
# - feature_cols : List[str]
# - task : Literal["classification", "regression"]
# - class_balance : Optional[Dict[str, float]] (if classification)
#
# Validators:
# - target_col must not be in feature_cols
# - if task == "classification", class_balance must be provided
#   and its values must sum to 1.0 (± 1e-6)
# - if task == "regression", class_balance must be None

from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Optional, List, Dict, Literal
import re

class DatasetSchema(BaseModel):
    pass  # to complete

# Tests
try:
    ds = DatasetSchema(
        name="churn_dataset",
        n_samples=5000,
        n_features=15,
        target_col="churn",
        feature_cols=["age", "tenure", "charges"],
        task="classification",
        class_balance={"0": 0.85, "1": 0.15},
    )
    print("OK:", ds.name, ds.task)
except Exception as e:
    print("Error:", e)

# Should fail: target in feature_cols
try:
    DatasetSchema(name="bad", n_samples=200, n_features=3,
                  target_col="age", feature_cols=["age", "tenure"],
                  task="regression")
except Exception as e:
    print(f"Expected error: {e.errors()[0]['msg'] if hasattr(e, 'errors') else e}")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# class DatasetSchema(BaseModel):
#     name: str
#     n_samples: int = Field(ge=100)
#     n_features: int = Field(ge=1)
#     target_col: str
#     feature_cols: List[str]
#     task: Literal["classification", "regression"]
#     class_balance: Optional[Dict[str, float]] = None
#
#     @field_validator("name")
#     @classmethod
#     def must_be_snake_case(cls, v):
#         if not re.match(r"^[a-z][a-z0-9_]*$", v):
#             raise ValueError(f"name must be snake_case, got '{v}'")
#         return v
#
#     @model_validator(mode="after")
#     def cross_validate(self):
#         if self.target_col in self.feature_cols:
#             raise ValueError("target_col must not be in feature_cols")
#         if self.task == "classification":
#             if self.class_balance is None:
#                 raise ValueError("class_balance required for classification")
#             total = sum(self.class_balance.values())
#             if abs(total - 1.0) > 1e-6:
#                 raise ValueError(f"class_balance must sum to 1.0, got {total:.4f}")
#         if self.task == "regression" and self.class_balance is not None:
#             raise ValueError("class_balance must be None for regression")
#         return self

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Dataclass vs Pydantic: know when to choose
# ════════════════════════════════════════════════════════
# For each case below, implement with the right
# approach (dataclass OR Pydantic) and justify in a comment.
#
# Case A: Internal configuration of a LightGBM model
#         (n_estimators, lr, num_leaves, features list)
#         → used only internally, not serialized from outside
#
# Case B: Input for a POST /predict endpoint
#         (age, income, contract_type, nb_products)
#         → data received from an HTTP client, must be validated

# CASE A
# to implement...

# CASE B
# to implement...

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# # CASE A → dataclass : config interne, pas de validation externe
# @dataclass
# class LGBMConfig:
#     n_estimators: int = 100
#     learning_rate: float = 0.05
#     num_leaves: int = 31
#     features: List[str] = field(default_factory=list)
#     def __post_init__(self):
#         if self.learning_rate <= 0: raise ValueError("lr must be > 0")
#
# # CASE B → Pydantic : données externes, validation stricte nécessaire
# class PredictRequest(BaseModel):
#     age: int = Field(ge=18, le=120)
#     income: float = Field(gt=0)
#     contract_type: Literal["month-to-month", "one-year", "two-year"]
#     nb_products: int = Field(ge=1, le=10, default=1)

---
## 2.3 Data manipulation — Pandas & Numpy

### Lesson — Pandas: essential operations

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500

df = pd.DataFrame({
    "customer_id":  range(1, n+1),
    "region":       np.random.choice(["IDF","PACA","AURA","HDF"], n),
    "age":          np.random.randint(18, 70, n),
    "income":       np.random.exponential(50000, n),
    "tenure":       np.random.randint(0, 60, n),
    "charges":      np.random.normal(65, 30, n).clip(10, 200),
    "nb_products":  np.random.randint(1, 6, n),
    "churn":        np.random.choice([0, 1], n, p=[0.8, 0.2]),
    "missing_col":  np.where(np.random.rand(n) < 0.15, np.nan, np.random.randn(n)),
    "date":         pd.date_range("2023-01-01", periods=n, freq="D"),
})

print("Shape:", df.shape)
print(df.dtypes)
print(df.describe().round(2))

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500
df = pd.DataFrame({
    "customer_id":  range(1, n+1),
    "region":       np.random.choice(["IDF","PACA","AURA","HDF"], n),
    "age":          np.random.randint(18, 70, n),
    "income":       np.random.exponential(50000, n),
    "tenure":       np.random.randint(0, 60, n),
    "charges":      np.random.normal(65, 30, n).clip(10, 200),
    "nb_products":  np.random.randint(1, 6, n),
    "churn":        np.random.choice([0, 1], n, p=[0.8, 0.2]),
    "missing_col":  np.where(np.random.rand(n) < 0.15, np.nan, np.random.randn(n)),
    "date":         pd.date_range("2023-01-01", periods=n, freq="D"),
})

# ── groupby + aggregation ──────────────────────────────────────
agg = (df.groupby("region")
         .agg(
             n_customers    = ("customer_id", "count"),
             avg_age        = ("age",     "mean"),
             avg_income     = ("income",  "mean"),
             churn_rate     = ("churn",   "mean"),
             avg_tenure     = ("tenure",  "mean"),
         )
         .round(2)
         .sort_values("churn_rate", ascending=False))
print("=== Groupby region ===")
print(agg)

# multiple groupby + transform (without reducing the DataFrame)
df["region_avg_income"] = df.groupby("region")["income"].transform("mean")
df["income_vs_region"]  = df["income"] / df["region_avg_income"]
print(f"income_vs_region stats: mean={df['income_vs_region'].mean():.3f}")

# ── merge / join ──────────────────────────────────────────────
region_info = pd.DataFrame({
    "region":     ["IDF","PACA","AURA","HDF"],
    "population": [12_000_000, 5_000_000, 8_000_000, 6_000_000],
    "tier":       ["A","B","A","B"],
})

df_merged = df.merge(region_info, on="region", how="left")
print(f"after merge: {df_merged.shape} (left join, no row lost)")

# ── fillna / astype / apply ───────────────────────────────────
# fillna with median (computed on train in prod → here on all data)
df["missing_filled"] = df["missing_col"].fillna(df["missing_col"].median())
print(f"missing avant: {df['missing_col'].isna().sum()}")
print(f"missing après: {df['missing_filled'].isna().sum()}")

# astype
df["churn_bool"] = df["churn"].astype(bool)
df["age_bin"]    = pd.cut(df["age"], bins=[0,25,35,50,70],
                           labels=["<25","25-35","35-50","50+"])

# apply — use sparingly (slow vs vectorization)
def classify_income(row):
    if row["income"] > 80000: return "high"
    if row["income"] > 30000: return "medium"
    return "low"

df["income_tier"] = df.apply(classify_income, axis=1)
print(df["income_tier"].value_counts())

### Lesson — Pandas: window functions, pivot, melt

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
# Time series
n = 200
ts = pd.DataFrame({
    "date":         pd.date_range("2023-01-01", periods=n, freq="D"),
    "region":       np.random.choice(["IDF","PACA"], n),
    "consumption":  100 + 10*np.sin(2*np.pi*np.arange(n)/7) + np.random.randn(n)*3,
})

# ── rolling / expanding / shift ───────────────────────────────
ts_sorted = ts.sort_values(["region","date"]).copy()

ts_sorted["rolling_7d"]  = (ts_sorted.groupby("region")["consumption"]
                                      .transform(lambda x: x.rolling(7, min_periods=1).mean()))
ts_sorted["rolling_std"] = (ts_sorted.groupby("region")["consumption"]
                                      .transform(lambda x: x.rolling(7, min_periods=1).std()))
ts_sorted["lag_7"]       = (ts_sorted.groupby("region")["consumption"]
                                      .transform(lambda x: x.shift(7)))
ts_sorted["cummax"]      = (ts_sorted.groupby("region")["consumption"]
                                      .transform(lambda x: x.expanding().max()))
ts_sorted["pct_change"]  = (ts_sorted.groupby("region")["consumption"]
                                      .transform(lambda x: x.pct_change()))

print("=== Time series features ===")
print(ts_sorted[["date","region","consumption","rolling_7d","lag_7","pct_change"]]
      .dropna().head(8).round(3).to_string(index=False))

# ── pivot_table ───────────────────────────────────────────────
n2 = 300
df2 = pd.DataFrame({
    "region":  np.random.choice(["IDF","PACA","AURA"], n2),
    "month":   np.random.choice(["Jan","Feb","Mar","Apr"], n2),
    "value":   np.random.exponential(100, n2),
    "churn":   np.random.choice([0,1], n2, p=[0.8,0.2]),
})

pivot = df2.pivot_table(values="value", index="region",
                         columns="month", aggfunc="mean").round(1)
print("=== Pivot table ===")
print(pivot)

# ── melt — wide to long ───────────────────────────────────────
wide = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "jan_revenue": [100, 200, 150],
    "feb_revenue": [120, 180, 160],
    "mar_revenue": [110, 210, 170],
})
long = wide.melt(id_vars="customer_id",
                  var_name="month",
                  value_name="revenue")
print("=== Melt (wide → long) ===")
print(long)

### Lesson — Numpy: vectorization and broadcasting

In [ ]:
import numpy as np

# ── Vectorization: avoid Python loops ─────────────────
import time

data = np.random.randn(1_000_000)

# ❌ Slow: Python loop
t0 = time.perf_counter()
result_loop = [x**2 for x in data]
t_loop = time.perf_counter() - t0

# ✅ Fast: vectorized operation
t0 = time.perf_counter()
result_vec = data ** 2
t_vec = time.perf_counter() - t0

print(f"Loop  : {t_loop*1000:.1f}ms")
print(f"Numpy : {t_vec*1000:.1f}ms")
print(f"Speedup: {t_loop/t_vec:.0f}x")

# ── Essential operations ───────────────────────────────────
X = np.random.randn(100, 5)

print("shape     :", X.shape)
print("mean/col  :", X.mean(axis=0).round(3))
print("std/col   :", X.std(axis=0).round(3))
print("max/row   :", X.max(axis=1)[:5].round(3))

# np.where — vectorized conditional
labels = np.where(X[:, 0] > 0, "positive", "negative")
print("np.where sample:", labels[:6])

# np.clip
clipped = np.clip(X, -1.5, 1.5)
print("clip [-1.5,1.5]: min={:.3f} max={:.3f}".format(clipped.min(), clipped.max()))

# np.log1p — safe log for values >= 0
incomes = np.array([0, 10, 100, 1000, 50000])
print("log1p:", np.log1p(incomes).round(3))

# ── Broadcasting ──────────────────────────────────────────────
# Vectorized standardization (z-score)
mean = X.mean(axis=0)   # shape (5,)
std  = X.std(axis=0)    # shape (5,)
X_std = (X - mean) / std   # broadcasting : (100,5) - (5,) = (100,5)
print("X_std mean:", X_std.mean(axis=0).round(6))   # ~0
print("X_std std :", X_std.std(axis=0).round(6))    # ~1

# Min-max scaling
X_mm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print("MinMax range: [{:.3f}, {:.3f}]".format(X_mm.min(), X_mm.max()))

# ── Advanced indexing ───────────────────────────────────────────
# Boolean indexing
y = np.random.choice([0, 1], 100)
X_pos = X[y == 1]   # rows of the positive class
print(f"Positive class: {X_pos.shape[0]} samples")

# Fancy indexing
idx = np.argsort(X[:, 0])[-10:]   # top 10 by column 0
print("Top 10 indices:", idx)

# np.unique with counts
classes, counts = np.unique(y, return_counts=True)
print("Class distribution:", dict(zip(classes, counts)))

### 🏋️ Exercises 2.3

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 — Full Pandas feature engineering
# ════════════════════════════════════════════════════════
# From the DataFrame df below, create the following features
# WITHOUT using a Python loop:
#
# a) income_log       = log1p(income)
# b) charges_per_tenure = charges / (tenure + 1)
# c) is_high_risk     = 1 if (tenure < 12) AND (nb_products == 1), else 0
# d) region_churn_rate = average churn rate of the customer's region
#                        (use groupby + transform)
# e) income_zscore    = z-score of income PER REGION
#                        (use groupby + transform)

import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000
df = pd.DataFrame({
    "customer_id": range(1, n+1),
    "region":      np.random.choice(["IDF","PACA","AURA","HDF"], n),
    "income":      np.random.exponential(50000, n),
    "tenure":      np.random.randint(0, 60, n),
    "charges":     np.random.normal(65, 30, n).clip(10, 200),
    "nb_products": np.random.randint(1, 6, n),
    "churn":       np.random.choice([0, 1], n, p=[0.8, 0.2]),
})

# to complete — each feature in one line
df["income_log"]         = None  # a)
df["charges_per_tenure"] = None  # b)
df["is_high_risk"]       = None  # c)
df["region_churn_rate"]  = None  # d)
df["income_zscore"]      = None  # e)

print(df[["income_log","charges_per_tenure","is_high_risk",
          "region_churn_rate","income_zscore"]].head(5).round(4))

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# df["income_log"]         = np.log1p(df["income"])
# df["charges_per_tenure"] = df["charges"] / (df["tenure"] + 1)
# df["is_high_risk"]       = ((df["tenure"] < 12) & (df["nb_products"] == 1)).astype(int)
# df["region_churn_rate"]  = df.groupby("region")["churn"].transform("mean")
# df["income_zscore"]      = df.groupby("region")["income"].transform(
#                                lambda x: (x - x.mean()) / x.std())

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCICE 2 — Window functions time series
# ════════════════════════════════════════════════════════
# Create these features on the time series ts,
# computed PER REGION (use groupby + transform):
#
# a) lag_7d          = consumption 7 days ago
# b) rolling_14d     = 14-day rolling average
# c) rolling_14d_std = 14-day rolling standard deviation
# d) pct_vs_rolling  = (consumption - rolling_14d) / rolling_14d
#                      (% deviation from the moving average)

import pandas as pd
import numpy as np

np.random.seed(42)
n = 300
ts = pd.DataFrame({
    "date":        pd.date_range("2023-01-01", periods=n, freq="D"),
    "region":      np.tile(["IDF","PACA","AURA"], n//3 + 1)[:n],
    "consumption": 100 + 10*np.sin(2*np.pi*np.arange(n)/7) + np.random.randn(n)*2,
})
ts = ts.sort_values(["region","date"]).copy()

# to complete
ts["lag_7d"]         = None  # a)
ts["rolling_14d"]    = None  # b)
ts["rolling_14d_std"]= None  # c)
ts["pct_vs_rolling"] = None  # d) — after computing b)

print(ts.dropna().head(8)[
    ["date","region","consumption","lag_7d","rolling_14d","pct_vs_rolling"]
].round(3).to_string(index=False))

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# g = ts.groupby("region")["consumption"]
# ts["lag_7d"]          = g.transform(lambda x: x.shift(7))
# ts["rolling_14d"]     = g.transform(lambda x: x.shift(1).rolling(14, min_periods=1).mean())
# ts["rolling_14d_std"] = g.transform(lambda x: x.shift(1).rolling(14, min_periods=1).std())
# ts["pct_vs_rolling"]  = (ts["consumption"] - ts["rolling_14d"]) / ts["rolling_14d"]

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Numpy: implement from scratch
# ════════════════════════════════════════════════════════
# Implement these 4 functions in pure numpy
# (no sklearn, no scipy), vectorized (no loop).

import numpy as np

# 3a. z_score_normalize(X) → each column: (x - mean) / std
def z_score_normalize(X: np.ndarray) -> np.ndarray:
    pass

# 3b. min_max_normalize(X) → each column: (x - min) / (max - min)
def min_max_normalize(X: np.ndarray) -> np.ndarray:
    pass

# 3c. cosine_similarity(a, b) → scalar between two vectors
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    pass

# 3d. softmax(x) → probability distribution over a vector
def softmax(x: np.ndarray) -> np.ndarray:
    pass  # note numerical stability: subtract max before exp

# Tests
X = np.array([[1.0, 200.0], [2.0, 300.0], [3.0, 400.0], [4.0, 500.0]])

Xz = z_score_normalize(X)
print("z-score mean:", Xz.mean(axis=0).round(6))   # [~0, ~0]
print("z-score std :", Xz.std(axis=0).round(6))    # [~1, ~1]

Xmm = min_max_normalize(X)
print("minmax min:", Xmm.min(axis=0))   # [0, 0]
print("minmax max:", Xmm.max(axis=0))   # [1, 1]

a, b = np.array([1.0, 0.0, 0.0]), np.array([1.0, 0.0, 0.0])
print("cosine same:", cosine_similarity(a, b))          # 1.0
print("cosine perp:", cosine_similarity(a, np.array([0.0, 1.0, 0.0])))  # 0.0

logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)
print("softmax:", probs.round(4))
print("sum:", probs.sum())   # 1.0

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def z_score_normalize(X):
#     return (X - X.mean(axis=0)) / X.std(axis=0)
#
# def min_max_normalize(X):
#     mn, mx = X.min(axis=0), X.max(axis=0)
#     return (X - mn) / np.where(mx - mn == 0, 1, mx - mn)
#
# def cosine_similarity(a, b):
#     return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)
#
# def softmax(x):
#     e = np.exp(x - x.max())   # stability trick
#     return e / e.sum()

In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 4 — Full Pandas pipeline: quality report
# ════════════════════════════════════════════════════════
# Write data_quality_report(df) that returns a DataFrame
# with one row per column and the following columns:
#   dtype, n_missing, pct_missing, n_unique,
#   mean (if numeric), std (if numeric),
#   min (if numeric), max (if numeric)
# Non-numeric columns have NaN for mean/std/min/max.
# Sorted by pct_missing descending.

import pandas as pd
import numpy as np

def data_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    pass  # to complete

np.random.seed(42)
n = 200
test_df = pd.DataFrame({
    "age":      np.where(np.random.rand(n)<0.05, np.nan, np.random.randint(18,70,n).astype(float)),
    "income":   np.random.exponential(50000, n),
    "region":   np.where(np.random.rand(n)<0.10, np.nan, np.random.choice(["IDF","PACA"], n)),
    "churn":    np.random.choice([0,1], n, p=[0.8,0.2]),
    "full_nan": np.full(n, np.nan),
})

report = data_quality_report(test_df)
if report is not None:
    print(report.to_string())

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def data_quality_report(df):
#     rows = []
#     for col in df.columns:
#         s = df[col]
#         is_num = pd.api.types.is_numeric_dtype(s)
#         row = {
#             "column":      col,
#             "dtype":       str(s.dtype),
#             "n_missing":   s.isna().sum(),
#             "pct_missing": round(s.isna().mean() * 100, 2),
#             "n_unique":    s.nunique(),
#             "mean": round(s.mean(), 4) if is_num else None,
#             "std":  round(s.std(),  4) if is_num else None,
#             "min":  round(s.min(),  4) if is_num else None,
#             "max":  round(s.max(),  4) if is_num else None,
#         }
#         rows.append(row)
#     return (pd.DataFrame(rows)
#               .set_index("column")
#               .sort_values("pct_missing", ascending=False))

---
## 📋 Module 2 Recap

| Topic | Test |
|---|---|
| Type annotations | Fully type a function with Optional, Union, Tuple |
| TypeVar | Write a generic function T→R |
| Protocol | Define an interface without inheritance |
| @dataclass | field(), __post_init__, frozen, replace() |
| Pydantic | field_validator, model_validator, Literal, ValidationError |
| dataclass vs Pydantic | Choose without hesitation based on context |
| groupby + agg | Multi-column aggregation in one expression |
| groupby + transform | Add a group-computed feature without reducing |
| merge | left/inner/outer, know what rows you lose |
| rolling + shift | Lag features and moving average per group |
| np.where | Vectorized conditional |
| Broadcasting | Column-wise standardization without loops |
| Softmax, cosine | Implement from scratch in numpy |
| data_quality_report | Generic quality report on a DataFrame |